# SCE-Net + DROCC для one-class compatibility (Fashion)

Ноутбук переводит задачу из pair-classification (`y in {0,1}`) в **one-class classification / anomaly detection**:
- в `train_pairs` и `val_pairs` только **позитивные пары** (`good/good-like`),
- в `test_pairs` есть и `good`, и `bad`,
- на инференсе используем **скалярный anomaly score** и подбираем порог на валидации/калибровочном сете.

Ключевая идея: сохраняем архитектурную логику SCE-Net (condition masks + condition weight branch + pair-context embedding), CLIP-признаки и image cache, но меняем objective на DROCC.


## 1) Что именно меняется относительно pair-classification

1. Убираем бинарный классификатор `good/bad` как основную цель обучения.
2. Вводим one-class objective:
   - стадия warm-up: учим scoring-head давать высокий score на позитивных парах,
   - стадия adversarial DROCC: генерируем синтетические «near-boundary negatives» вокруг позитивов и отталкиваем их вниз.
3. На тесте считаем `p_normal = sigmoid(score)`, а `anomaly_score = 1 - p_normal`.
4. Порог выбираем по целевой метрике (F1 / balanced accuracy / precision@recall) на отложенном наборе с разметкой.


In [ ]:
import os
import math
import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoProcessor, AutoModel
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve


## 2) Конфиг (сохраняем структуру и ключевые поля, меняем `hf_model_name`)


In [ ]:
@dataclass
class Config:
    seed: int = 42
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Пути
    train_pairs_path: str = 'data/train_pairs_positive.csv'
    val_pairs_path: str = 'data/val_pairs_positive.csv'
    test_pairs_path: str = 'data/test_pairs_labeled.csv'
    items_path: str = 'data/items_catalog.csv'
    image_root: str = 'data/images'

    # Encoder
    hf_model_name: str = 'patrickjohncyh/fashion-clip'
    clip_trainable: bool = False

    # SCE-Net
    embed_dim: int = 512
    condition_masks: int = 8
    weight_hidden: int = 512
    mask_l1_lambda: float = 1e-4
    feat_l2_lambda: float = 1e-6

    # DROCC
    epochs: int = 20
    warmup_epochs: int = 3
    batch_size: int = 64
    num_workers: int = 4
    lr: float = 1e-4
    weight_decay: float = 1e-4

    drocc_radius: float = 1.0
    drocc_gamma: float = 2.0
    drocc_steps: int = 5
    drocc_step_size: float = 0.1
    drocc_lambda: float = 1.0

    # инференс
    threshold_metric: str = 'f1'

cfg = Config()
print(asdict(cfg))


## 3) Детально: DROCC и как он ложится на SCE-Net пары


### Интуиция DROCC (Deep Robust One-Class Classification, 2020)

DROCC учит decision boundary вокруг **плотной области нормальных объектов**. Вместо случайных отрицательных примеров алгоритм строит **adversarial negatives** рядом с нормальными данными на сферической оболочке (`r` .. `gamma*r`) и заставляет модель отделять их как anomalous.

### В терминах нашей задачи compatibility

- Объектом one-class является **пара товаров** `(sku1, sku2)`.
- Позитивная пара проходит через SCE-Net и даёт pair-context representation `z`.
- Scoring-head `f(z)` выдаёт логит «normality».
- Для каждой позитивной пары ищем adversarial точку `z_adv` в окрестности `z`:
  - максимизируем loss «сделать её тоже normal»,
  - проецируем на annulus с радиусами `[r, gamma*r]`.
- Затем в основном шаге обучения:
  - `z` push up (normal),
  - `z_adv` push down (anomaly).

Это снижает эффект «перемудрённых негативов из эвристик», потому что negative signal появляется из геометрии нормального manifold, а не из ручной генерации плохих пар.


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)


## 4) Данные и RAM-cache изображений (переиспользуем базовый пайплайн)


In [ ]:
items_df = pd.read_csv(cfg.items_path)
train_df = pd.read_csv(cfg.train_pairs_path)  # только позитивы
val_df = pd.read_csv(cfg.val_pairs_path)      # только позитивы
test_df = pd.read_csv(cfg.test_pairs_path)    # good/bad для оценки и порога

id2path = dict(zip(items_df['sku'], items_df['image_path']))

processor = AutoProcessor.from_pretrained(cfg.hf_model_name)

image_cache: Dict[str, torch.Tensor] = {}

def load_image_tensor(sku: str):
    if sku in image_cache:
        return image_cache[sku]
    path = Path(cfg.image_root) / id2path[sku]
    img = Image.open(path).convert('RGB')
    px = processor(images=img, return_tensors='pt')['pixel_values'][0]
    image_cache[sku] = px
    return px


In [ ]:
class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, with_labels: bool):
        self.df = df.reset_index(drop=True)
        self.with_labels = with_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        s1, s2 = str(r['sku1']), str(r['sku2'])
        out = {
            'sku1': s1,
            'sku2': s2,
            'img1': load_image_tensor(s1),
            'img2': load_image_tensor(s2),
        }
        if self.with_labels:
            out['y'] = float(r['y'])
        return out


def collate_fn(batch):
    img1 = torch.stack([b['img1'] for b in batch])
    img2 = torch.stack([b['img2'] for b in batch])
    out = {'img1': img1, 'img2': img2}
    if 'y' in batch[0]:
        out['y'] = torch.tensor([b['y'] for b in batch], dtype=torch.float32)
    return out


## 5) SCE-Net encoder (condition masks + condition weight branch) + DROCC scoring head


In [ ]:
class SCENetOneClass(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.backbone = AutoModel.from_pretrained(cfg.hf_model_name)
        if not cfg.clip_trainable:
            for p in self.backbone.parameters():
                p.requires_grad = False

        d = cfg.embed_dim
        m = cfg.condition_masks
        self.condition_masks = nn.Parameter(torch.randn(m, d) * 0.02)
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * d, cfg.weight_hidden), nn.ReLU(),
            nn.Linear(cfg.weight_hidden, m)
        )
        self.scorer = nn.Sequential(
            nn.Linear(d, d), nn.ReLU(), nn.Linear(d, 1)
        )

    def encode_items(self, pixel_values):
        out = self.backbone.get_image_features(pixel_values=pixel_values)
        out = F.normalize(out, p=2, dim=-1)
        return out

    def pair_embedding(self, v1, v2):
        w = torch.softmax(self.weight_branch(torch.cat([v1, v2], dim=-1)), dim=-1)  # [B, M]
        masked_1 = v1.unsqueeze(1) * self.condition_masks.unsqueeze(0)               # [B, M, D]
        masked_2 = v2.unsqueeze(1) * self.condition_masks.unsqueeze(0)
        e1 = torch.bmm(w.unsqueeze(1), masked_1).squeeze(1)                           # [B, D]
        e2 = torch.bmm(w.unsqueeze(1), masked_2).squeeze(1)
        z = F.normalize((e1 + e2) / 2.0, p=2, dim=-1)
        return z

    def forward(self, img1, img2):
        v1 = self.encode_items(img1)
        v2 = self.encode_items(img2)
        z = self.pair_embedding(v1, v2)
        logit = self.scorer(z).squeeze(-1)
        return logit, z


## 6) DROCC loss и adversarial generation в embedding-space


In [ ]:
def drocc_generate_adv(z, model, steps=5, step_size=0.1, r=1.0, gamma=2.0):
    z_adv = z.detach() + 0.001 * torch.randn_like(z)
    z_adv.requires_grad_(True)

    for _ in range(steps):
        logit_adv = model.scorer(z_adv).squeeze(-1)
        # maximize normal-score for adversarial crafting
        adv_obj = F.binary_cross_entropy_with_logits(logit_adv, torch.ones_like(logit_adv))
        grad = torch.autograd.grad(adv_obj, z_adv, retain_graph=False, create_graph=False)[0]
        z_adv = z_adv.detach() + step_size * F.normalize(grad, dim=-1)

        delta = z_adv - z
        dist = delta.norm(p=2, dim=-1, keepdim=True).clamp_min(1e-8)
        dist_proj = dist.clamp(min=r, max=gamma * r)
        z_adv = (z + delta / dist * dist_proj).detach()
        z_adv.requires_grad_(True)

    return z_adv.detach()


def one_class_loss(model, z, cfg: Config):
    # positives
    logit_pos = model.scorer(z).squeeze(-1)
    pos_loss = F.binary_cross_entropy_with_logits(logit_pos, torch.ones_like(logit_pos))

    # drocc adversarial negatives
    z_adv = drocc_generate_adv(
        z, model,
        steps=cfg.drocc_steps,
        step_size=cfg.drocc_step_size,
        r=cfg.drocc_radius,
        gamma=cfg.drocc_gamma,
    )
    logit_adv = model.scorer(z_adv).squeeze(-1)
    neg_loss = F.binary_cross_entropy_with_logits(logit_adv, torch.zeros_like(logit_adv))

    l1 = model.condition_masks.abs().mean()
    l2 = z.pow(2).mean()
    total = pos_loss + cfg.drocc_lambda * neg_loss + cfg.mask_l1_lambda * l1 + cfg.feat_l2_lambda * l2
    return total, {'pos_loss': pos_loss.item(), 'neg_loss': neg_loss.item()}


In [ ]:
train_loader = DataLoader(PairDataset(train_df, with_labels=False), batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, collate_fn=collate_fn)
val_loader = DataLoader(PairDataset(val_df, with_labels=False), batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=collate_fn)
test_loader = DataLoader(PairDataset(test_df, with_labels=True), batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=collate_fn)

model = SCENetOneClass(cfg).to(cfg.device)
opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.lr, weight_decay=cfg.weight_decay)


## 7) Тренировка (warm-up + DROCC phase)


In [ ]:
for epoch in range(cfg.epochs):
    model.train()
    losses = []
    for batch in tqdm(train_loader, desc=f'train epoch {epoch}'):
        img1 = batch['img1'].to(cfg.device)
        img2 = batch['img2'].to(cfg.device)
        opt.zero_grad()
        logit, z = model(img1, img2)

        if epoch < cfg.warmup_epochs:
            loss = F.binary_cross_entropy_with_logits(logit, torch.ones_like(logit))
        else:
            loss, _ = one_class_loss(model, z, cfg)

        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f'epoch={epoch} train_loss={np.mean(losses):.4f}')


## 8) Оценка на test и подбор порога


In [ ]:
@torch.no_grad()
def predict_scores(loader):
    model.eval()
    ys, scores = [], []
    for batch in tqdm(loader, desc='infer'):
        img1 = batch['img1'].to(cfg.device)
        img2 = batch['img2'].to(cfg.device)
        logit, _ = model(img1, img2)
        p_normal = torch.sigmoid(logit).cpu().numpy()
        anom = 1.0 - p_normal
        scores.extend(anom.tolist())
        if 'y' in batch:
            ys.extend(batch['y'].numpy().tolist())
    return np.array(ys), np.array(scores)

y_true, anom_score = predict_scores(test_loader)
# в test y=1 good, y=0 bad => для anomaly цели инвертируем
y_anom = 1 - y_true
print('ROC-AUC(anomaly):', roc_auc_score(y_anom, anom_score))
print('PR-AUC(anomaly):', average_precision_score(y_anom, anom_score))

prec, rec, thr = precision_recall_curve(y_anom, anom_score)
f1 = 2 * prec * rec / (prec + rec + 1e-8)
best_i = np.nanargmax(f1)
best_thr = thr[max(0, best_i - 1)] if len(thr) > 0 else 0.5
print('best threshold:', float(best_thr), 'best F1:', float(np.nanmax(f1)))


## 9) Практические рекомендации именно для вашего кейса

1. **Не выбрасывайте полностью bad-пары**: оставьте их только для evaluation/threshold calibration, но не для основного train objective.
2. **Критично подобрать `drocc_radius`**:
   - слишком маленький -> слабая граница,
   - слишком большой -> модель начинает резать валидные «редкие» good-пары.
3. **Стабилизация**:
   - первые `2-5` эпох только positive BCE,
   - затем включать DROCC.
4. **Диагностика «белая футболка + джинсы = bad»**:
   - проверить распределение anomaly_score по базовым canonical-парам,
   - добавить отдельный мониторинг по категориям (`top+bottom`, `dress+shoes`).
5. **Калибровка**:
   - глобальный порог + (опционально) category-aware пороги.
6. **Hard positive mining**:
   - если есть мета-инфа про «сильные образы», можно reweight позитивы, чтобы manifold строился вокруг действительно качественных комбинаций.
